# KADMON Optuna — K-Means + Sinkhorn

Le classement utilise `global_distance_mm`. L'objectif minimise le rapport entre distance intra-identité et distance moyenne inter-identité.

## Architecture CPU/GPU

- K-Means et préparation : parallélisme multi-CPU
- MDF, matrices de coût, normalisation et OT : GPU
- Optuna : un seul essai GPU à la fois

Le GPU n'est activé qu'après validation numérique contre DIPY/POT.

In [1]:
from pathlib import Path
import hashlib, json, os, sys, warnings
from time import perf_counter
import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from joblib import Parallel, delayed
from optuna.trial import TrialState

NOTEBOOK_DIR = Path.cwd().resolve()
KADMON_ROOT = (NOTEBOOK_DIR / '../..').resolve()
BUNDLES_DIR = NOTEBOOK_DIR.parent / 'bundles'
STUDY_PATH = NOTEBOOK_DIR / 'studies' / 'optuna_reid.sqlite3'
COMPRESSION_CACHE_DIR = NOTEBOOK_DIR / 'cache' / 'kmeans_cpu'
if str(KADMON_ROOT) not in sys.path: sys.path.insert(0, str(KADMON_ROOT))
from kadmon.compression import compress_kmeans
from kadmon.gpu import (
    evaluate_pair_cpu as evaluate_sinkhorn_pair_cpu,
    evaluate_pair_gpu as evaluate_sinkhorn_pair_gpu,
    release_gpu_memory,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
EXPERIMENT_NAME, COMPRESSION, TRANSPORT, N_TRIALS = 'kmeans_sinkhorn', 'kmeans', 'sinkhorn', 80
GPU_DTYPE, GPU_MDF_BATCH_SIZE = torch.float32, 128
GPU_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_BACKEND_VALIDATED = False
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(GPU_DEVICE)
    print(f'[GPU] backend=PyTorch/POT, device={torch.cuda.get_device_name(GPU_DEVICE)}')
    print(f'[GPU] mémoire={free/2**30:.2f}/{total/2**30:.2f} Gio, dtype={GPU_DTYPE}')
    print('[GPU] MDF/coût=torch_mdf_pairwise_batched; OT=ot.sinkhorn backend torch')
else:
    warnings.warn('CUDA indisponible : fallback CPU explicite.', RuntimeWarning)


Info: some functions in tractosearch.resampling are faster when 'numba' is installed
[GPU] backend=PyTorch/POT, device=NVIDIA GeForce RTX 5070 Ti
[GPU] mémoire=15.28/15.51 Gio, dtype=torch.float32
[GPU] MDF/coût=torch_mdf_pairwise_batched; OT=ot.sinkhorn backend torch


## Données et compression K-Means CPU mise en cache

In [2]:
SUBJECT_PAIRS = {'103818':'103818_re','135528':'135528_re','143325':'143325_re','177746':'177746_re','194140':'194140_re','250427':'250427_re','433839':'433839_re','627549':'627549_re','783462':'783462_re','861456':'861456_re'}
REFERENCE_SUBJECT = '103818'
INTRA_IDENTITY_SUBJECT = SUBJECT_PAIRS[REFERENCE_SUBJECT]
COMPARISON_SUBJECTS = tuple(SUBJECT_PAIRS.values())
SUBJECTS = (REFERENCE_SUBJECT, *COMPARISON_SUBJECTS)
N_POINTS, SEED, BUNDLES_TO_RUN = 12, 42, None
N_JOBS_CPU, CACHE_VERSION = min(8, os.cpu_count() or 1), 'kmeans-oriented-v1'

def index_subject(subject):
    suffix = f'_{N_POINTS}mpts_rasmm.npy'
    paths = sorted((BUNDLES_DIR / subject / 'nn_8mm').glob(f'*{suffix}'))
    return {p.name[:-len(suffix)]: p for p in paths}

SUBJECT_FILES = {s:index_subject(s) for s in SUBJECTS}
bundle_names = sorted(set.intersection(*(set(x) for x in SUBJECT_FILES.values())))
if BUNDLES_TO_RUN is not None: bundle_names = [x for x in bundle_names if x in BUNDLES_TO_RUN]
if not bundle_names: raise RuntimeError('Aucun bundle commun.')
bundle_cache, compression_cache = {}, {}

def load_bundle(subject, bundle_name):
    key = (subject, bundle_name)
    if key not in bundle_cache:
        value = np.load(SUBJECT_FILES[subject][bundle_name], mmap_mode='r')
        if value.ndim != 3 or value.shape[1:] != (N_POINTS,3) or not len(value) or not np.isfinite(value).all():
            raise ValueError(f'Bundle invalide: {key}, {value.shape}')
        bundle_cache[key] = value
    return bundle_cache[key]

def effective_parameters(subject, bundle_name, parameters):
    result = dict(parameters)
    result['n_clusters'] = min(int(result['n_clusters']), len(load_bundle(subject,bundle_name)))
    return result

def compression_key(subject, bundle_name, parameters):
    path, stat = SUBJECT_FILES[subject][bundle_name], SUBJECT_FILES[subject][bundle_name].stat()
    metadata = {'version':CACHE_VERSION,'subject':subject,'bundle':bundle_name,'path':str(path.resolve()),'size':stat.st_size,'mtime_ns':stat.st_mtime_ns,'parameters':sorted(parameters.items()),'n_points':N_POINTS}
    digest = hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()[:24]
    return digest, metadata

def compress_one_cpu(subject, bundle_name, parameters):
    digest, metadata = compression_key(subject,bundle_name,parameters)
    path = COMPRESSION_CACHE_DIR / f'{digest}.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            if str(z['metadata']) == json.dumps(metadata,sort_keys=True):
                return (subject,bundle_name),(z['representatives'],z['weights']),True
    reps, weights = compress_kmeans(np.asarray(load_bundle(subject,bundle_name),dtype=np.float64),**parameters)
    COMPRESSION_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    temporary = path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary,representatives=reps,weights=weights,metadata=json.dumps(metadata,sort_keys=True))
    os.replace(temporary,path)
    return (subject,bundle_name),(reps,weights),False

def prepare_kmeans_compressions_cpu(parameters, selected_bundles=None):
    selected = tuple(bundle_names if selected_bundles is None else selected_bundles)
    tasks, hits = [], 0
    for bundle_name in selected:
        for subject in SUBJECTS:
            effective = effective_parameters(subject,bundle_name,parameters)
            digest,_ = compression_key(subject,bundle_name,effective)
            key = (subject,bundle_name,digest)
            if key in compression_cache: hits += 1
            else: tasks.append((subject,bundle_name,effective,key))
    computed = Parallel(n_jobs=min(N_JOBS_CPU,len(tasks)),backend='loky')(delayed(compress_one_cpu)(s,b,p) for s,b,p,_ in tasks) if tasks else []
    disk_hits = 0
    for (_,distribution,disk_hit),(_,_,_,key) in zip(computed,tasks):
        compression_cache[key] = distribution; disk_hits += int(disk_hit)
    result = {}
    for bundle_name in selected:
        for subject in SUBJECTS:
            effective = effective_parameters(subject,bundle_name,parameters)
            digest,_ = compression_key(subject,bundle_name,effective)
            result[(subject,bundle_name)] = compression_cache[(subject,bundle_name,digest)]
    print(f'[CPU] K-Means compression: {hits+disk_hits} cache hit(s), {len(tasks)-disk_hits} calcul(s)')
    return result

print(f'Données={BUNDLES_DIR}; bundles={len(bundle_names)}; workers CPU={N_JOBS_CPU}')


Données=/home/colin/Tractographie/KADMON/notebooks/bundles; bundles=31; workers CPU=8


## MDF batché et Sinkhorn débiaisé GPU

In [ ]:
def evaluate_pair_gpu(source_distribution, target_distribution, parameters, log=False, gpu_cache=None, return_cost=False):
    return evaluate_sinkhorn_pair_gpu(
        source_distribution,
        target_distribution,
        parameters,
        device=GPU_DEVICE,
        dtype=GPU_DTYPE,
        batch_size=GPU_MDF_BATCH_SIZE,
        self_cost_cache=gpu_cache,
        return_cost=return_cost,
        log=log,
    )


def evaluate_pair_cpu(source_distribution, target_distribution, parameters):
    return evaluate_sinkhorn_pair_cpu(
        source_distribution, target_distribution, parameters
    )


release_trial_gpu = release_gpu_memory


## Validation CPU/GPU obligatoire
Le plus petit bundle commun est testé sur les dix candidats; coûts, OT, poids et ratio objectif sont comparés.

In [4]:
VALIDATION_KMEANS = {'n_clusters':25,'max_iter':300,'tol':1e-4,'seed':SEED}
VALIDATION_OT = {'epsilon':.75,'max_iter':2000,'stop_threshold':1e-6,'reject_threshold':1e-5}
GPU_VALIDATION_RESULTS = pd.DataFrame()

def validate_gpu_backend():
    global GPU_BACKEND_VALIDATED,GPU_VALIDATION_RESULTS
    if not torch.cuda.is_available(): return False
    pilot = min(bundle_names,key=lambda name:sum(len(load_bundle(s,name)) for s in SUBJECTS))
    distributions = prepare_kmeans_compressions_cpu(VALIDATION_KMEANS,[pilot])
    rows,cpu_dist,gpu_dist = [],[],[]
    for i,candidate in enumerate(COMPARISON_SUBJECTS):
        source = distributions[(REFERENCE_SUBJECT,pilot)]; target = distributions[(candidate,pilot)]
        cpu = evaluate_pair_cpu(source,target,VALIDATION_OT); gpu = evaluate_pair_gpu(source,target,VALIDATION_OT,log=(i==0),return_cost=True)
        difference = np.abs(cpu['cost']-gpu['cost']); cpu_dist.append(cpu['global_distance_mm']); gpu_dist.append(gpu['global_distance_mm'])
        rows.append({'bundle':pilot,'candidate':candidate,'cost_max_abs':difference.max(),'cost_mean_abs':difference.mean(),'ot_distance_abs':abs(cpu['global_distance_mm']-gpu['global_distance_mm']),'ot_objective_abs':abs(cpu['objective']-gpu['objective']),'source_weight_max_abs':np.max(np.abs(source[1]-gpu['weights'][0])),'target_weight_max_abs':np.max(np.abs(target[1]-gpu['weights'][1]))})
    intra = COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT)
    cpu_ratio = cpu_dist[intra]/np.delete(cpu_dist,intra).mean(); gpu_ratio = gpu_dist[intra]/np.delete(gpu_dist,intra).mean()
    GPU_VALIDATION_RESULTS = pd.DataFrame(rows); display(GPU_VALIDATION_RESULTS)
    print(f'[validation] coût max={GPU_VALIDATION_RESULTS.cost_max_abs.max():.3e}, moyen={GPU_VALIDATION_RESULTS.cost_mean_abs.mean():.3e}')
    print(f'[validation] OT max={GPU_VALIDATION_RESULTS.ot_distance_abs.max():.3e}; poids max={GPU_VALIDATION_RESULTS[["source_weight_max_abs","target_weight_max_abs"]].to_numpy().max():.3e}')
    print(f'[validation] ratio CPU={cpu_ratio:.8f}, GPU={gpu_ratio:.8f}, diff={abs(cpu_ratio-gpu_ratio):.3e}')
    GPU_BACKEND_VALIDATED = bool(GPU_VALIDATION_RESULTS.cost_max_abs.max()<=2e-4 and np.allclose(cpu_dist,gpu_dist,rtol=5e-4,atol=1e-5) and np.isclose(cpu_ratio,gpu_ratio,rtol=5e-4,atol=1e-5) and GPU_VALIDATION_RESULTS[['source_weight_max_abs','target_weight_max_abs']].to_numpy().max()==0)
    if not GPU_BACKEND_VALIDATED: warnings.warn('Validation CPU/GPU échouée: fallback CPU.',RuntimeWarning)
    else: print('[GPU] Validation réussie; backend GPU activé.')
    release_trial_gpu(); return GPU_BACKEND_VALIDATED

validate_gpu_backend()


Info: some functions in tractosearch.resampling are faster when 'numba' is installed
Info: some functions in tractosearch.resampling are faster when 'numba' is installed
Info: some functions in tractosearch.resampling are faster when 'numba' is installedInfo: some functions in tractosearch.resampling are faster when 'numba' is installed

Info: some functions in tractosearch.resampling are faster when 'numba' is installed
Info: some functions in tractosearch.resampling are faster when 'numba' is installed
Info: some functions in tractosearch.resampling are faster when 'numba' is installed
Info: some functions in tractosearch.resampling are faster when 'numba' is installed
[CPU] K-Means compression: 11 cache hit(s), 0 calcul(s)
[GPU] Cost matrix: shape=(25, 25), dtype=torch.float32
[GPU] OT backend: PyTorch/POT


,bundle,candidate,cost_max_abs,cost_mean_abs,ot_distance_abs,ot_objective_abs,source_weight_max_abs,target_weight_max_abs
0,tractosearch_nn_8_0mm_all_IFOF_R_m,103818_re,0.000004,7.179260e-07,2.302384e-07,1.448623e-08,0.0,0.0
1,tractosearch_nn_8_0mm_all_IFOF_R_m,135528_re,0.000004,7.045746e-07,1.811021e-07,3.240868e-08,0.0,0.0
2,tractosearch_nn_8_0mm_all_IFOF_R_m,143325_re,0.000004,8.659363e-07,5.327592e-07,3.378219e-09,0.0,0.0
3,tractosearch_nn_8_0mm_all_IFOF_R_m,177746_re,0.000004,7.026672e-07,2.372846e-07,3.663708e-08,0.0,0.0
4,tractosearch_nn_8_0mm_all_IFOF_R_m,194140_re,0.000004,8.678436e-07,5.320045e-08,3.391296e-08,0.0,0.0
5,tractosearch_nn_8_0mm_all_IFOF_R_m,250427_re,0.000004,7.217407e-07,5.757045e-07,2.871318e-08,0.0,0.0
6,tractosearch_nn_8_0mm_all_IFOF_R_m,433839_re,0.000004,7.751465e-07,1.764154e-07,7.326077e-09,0.0,0.0
7,tractosearch_nn_8_0mm_all_IFOF_R_m,627549_re,0.000004,1.052475e-06,4.572664e-07,1.213050e-07,0.0,0.0
8,tractosearch_nn_8_0mm_all_IFOF_R_m,783462_re,0.000004,8.937836e-07,5.100548e-07,4.346531e-08,0.0,0.0
9,tractosearch_nn_8_0mm_all_IFOF_R_m,861456_re,0.000004,7.827759e-07,2.970360e-07,5.539688e-08,0.0,0.0


[validation] coût max=3.815e-06, moyen=8.085e-07
[validation] OT max=5.757e-07; poids max=0.000e+00
[validation] ratio CPU=0.91968138, GPU=0.91968138, diff=2.122e-09
[GPU] Validation réussie; backend GPU activé.


True

## Objectif Optuna séquentiel sur GPU

In [5]:
def sample_parameters(trial):
    return ({'n_clusters':trial.suggest_int('n_clusters',25,300,step=5),'max_iter':300,'tol':1e-4,'seed':SEED},{'epsilon':trial.suggest_float('epsilon',1e-3,1.,log=True),'max_iter':2000,'stop_threshold':1e-6,'reject_threshold':1e-5})

def evaluate_bundle(bundle_name,distributions,parameters,use_gpu):
    pair_metrics=[]; gpu_cache={}
    for i,candidate in enumerate(COMPARISON_SUBJECTS):
        function = evaluate_pair_gpu if use_gpu else evaluate_pair_cpu
        kwargs = {'log':i==0,'gpu_cache':gpu_cache} if use_gpu else {}
        value = function(distributions[(REFERENCE_SUBJECT,bundle_name)],distributions[(candidate,bundle_name)],parameters,**kwargs)
        pair_metrics.append(value)
    gpu_cache.clear()
    distances=np.asarray([row['global_distance_mm'] for row in pair_metrics]); intra=COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT); inter=np.delete(distances,intra); order=np.argsort(distances,kind='stable')
    return {'bundle':bundle_name,'intra_inter_ratio':distances[intra]/inter.mean(),'intra_identity_top1_success':np.argmin(distances)==intra,'intra_identity_rank':np.flatnonzero(order==intra)[0]+1,'intra_inter_separation_margin_mm':inter.min()-distances[intra],'intra_identity_distance_mm':distances[intra],'mean_inter_identity_distance_mm':inter.mean(),'mean_displacement_mm':np.mean([row['mean_mm'] for row in pair_metrics]),'mean_transported_mass':np.mean([row['mass'] for row in pair_metrics]),'mean_n_representatives':np.mean([len(distributions[(s,bundle_name)][0]) for s in SUBJECTS]),'n_clusters_effective':min(len(distributions[(s,bundle_name)][0]) for s in SUBJECTS)}

def evaluate_trial_gpu(trial,compression_parameters,transport_parameters):
    distributions=prepare_kmeans_compressions_cpu(compression_parameters)
    use_gpu=GPU_BACKEND_VALIDATED
    if not use_gpu: warnings.warn('[CPU fallback] CUDA absent ou validation échouée; évaluation CPU séquentielle.',RuntimeWarning)
    results=[evaluate_bundle(name,distributions,transport_parameters,use_gpu) for name in bundle_names]
    print(f'[GPU] Trial {trial.number} completed' if use_gpu else f'[CPU fallback] Trial {trial.number} completed')
    return pd.DataFrame(results),use_gpu

def objective(trial):
    started=perf_counter(); compression_parameters,transport_parameters=sample_parameters(trial)
    try: bundle_metrics,use_gpu=evaluate_trial_gpu(trial,compression_parameters,transport_parameters)
    except RuntimeError as exc:
        if 'Sinkhorn' in str(exc) and 'convergé' in str(exc): raise optuna.TrialPruned(str(exc)) from exc
        raise
    finally: release_trial_gpu()
    valid_mask=bundle_metrics.intra_identity_top1_success.astype(bool)
    aggregates={'reid_valid_bundles':bundle_metrics.loc[valid_mask,'bundle'].tolist(),'reid_failed_bundles':bundle_metrics.loc[~valid_mask,'bundle'].tolist(),'mean_intra_inter_ratio':bundle_metrics.intra_inter_ratio.mean(),'median_intra_inter_ratio':bundle_metrics.intra_inter_ratio.median(),'intra_identity_top1_accuracy':bundle_metrics.intra_identity_top1_success.mean(),'mean_intra_identity_rank':bundle_metrics.intra_identity_rank.mean(),'mean_intra_inter_separation_margin_mm':bundle_metrics.intra_inter_separation_margin_mm.mean(),'mean_intra_identity_distance_mm':bundle_metrics.intra_identity_distance_mm.mean(),'mean_inter_identity_distance_mm':bundle_metrics.mean_inter_identity_distance_mm.mean(),'mean_displacement_mm':bundle_metrics.mean_displacement_mm.mean(),'mean_transported_mass':bundle_metrics.mean_transported_mass.mean(),'mean_n_representatives':bundle_metrics.mean_n_representatives.mean(),'n_bundles':len(bundle_metrics),'n_comparisons':len(bundle_metrics)*len(COMPARISON_SUBJECTS),'elapsed_s':perf_counter()-started,'n_clusters_effective_min':bundle_metrics.n_clusters_effective.min(),'n_clusters_effective_max':bundle_metrics.n_clusters_effective.max(),'backend':'PyTorch/POT CUDA' if use_gpu else 'CPU fallback'}
    for name,value in aggregates.items(): trial.set_user_attr(name,float(value) if isinstance(value,(np.floating,np.integer)) else value)
    return float(aggregates['mean_intra_inter_ratio'])


## Optimisation et résultats
`n_jobs=1` empêche explicitement plusieurs essais CUDA simultanés.

In [6]:
STUDY_PATH.parent.mkdir(parents=True,exist_ok=True)
study=optuna.create_study(study_name=EXPERIMENT_NAME,storage=f'sqlite:///{STUDY_PATH}',load_if_exists=True,direction='minimize',sampler=optuna.samplers.TPESampler(seed=SEED))
completed=sum(t.state==TrialState.COMPLETE for t in study.trials)
remaining=max(0,N_TRIALS-completed)
max_additional_attempts=max(50,4*remaining) if remaining else 0
attempts=0
print(f'Étude: {completed}/{N_TRIALS} essais COMPLETE; cible restante={remaining}.')
while completed < N_TRIALS and attempts < max_additional_attempts:
    batch=min(N_TRIALS-completed,max_additional_attempts-attempts)
    study.optimize(objective,n_trials=batch,n_jobs=1,show_progress_bar=True,gc_after_trial=False)
    attempts += batch
    completed=sum(t.state==TrialState.COMPLETE for t in study.trials)
    print(f'Progression: {completed}/{N_TRIALS} essais COMPLETE après {attempts} nouvelle(s) tentative(s).')
if completed < N_TRIALS:
    warnings.warn(f'Cible non atteinte: {completed}/{N_TRIALS} essais COMPLETE après {attempts} tentatives; relancer la cellule pour continuer.',RuntimeWarning)
trials_df=study.trials_dataframe(attrs=('number','value','params','user_attrs','state')); display(trials_df.sort_values('value').head(10))
best=study.best_trial; display(pd.Series({'experiment':EXPERIMENT_NAME,'trial':best.number,'mean_intra_inter_ratio':best.value,**best.params,**best.user_attrs},name='meilleur essai').to_frame())


Étude: 80/80 essais COMPLETE; cible restante=0.


,number,value,params_epsilon,params_n_clusters,user_attrs_backend,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,user_attrs_mean_intra_identity_distance_mm,...,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_clusters_effective_max,user_attrs_n_clusters_effective_min,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
146,146,0.441694,0.012951,155,PyTorch/POT CUDA,76.024927,0.935484,6.244780,7.703292,3.402041,...,153.739003,1.0,0.412228,31.0,155.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
132,132,0.442375,0.013497,165,PyTorch/POT CUDA,78.261174,0.935484,6.236730,7.718507,3.413396,...,163.466276,1.0,0.406483,31.0,165.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
118,118,0.443212,0.012042,225,PyTorch/POT CUDA,77.495486,0.935484,6.250991,7.672580,3.402349,...,221.431085,1.0,0.409001,31.0,225.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
148,148,0.443369,0.013088,185,PyTorch/POT CUDA,77.514345,0.935484,6.238193,7.705184,3.417541,...,182.841642,1.0,0.402332,31.0,185.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
139,139,0.443602,0.012603,180,PyTorch/POT CUDA,76.435882,0.935484,6.246904,7.688361,3.409970,...,178.002933,1.0,0.400190,31.0,180.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
74,74,0.443683,0.012519,145,PyTorch/POT CUDA,73.667057,0.935484,6.252374,7.691395,3.409700,...,143.991202,1.0,0.403641,31.0,145.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
140,140,0.443711,0.012786,180,PyTorch/POT CUDA,75.580287,0.935484,6.244268,7.693680,3.413344,...,178.002933,1.0,0.400300,31.0,180.0,45.0,310.0,"[tractosearch_nn_8_0mm_all_CST_L_m, tractosear...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
48,48,0.443953,0.012512,280,PyTorch/POT CUDA,79.602713,0.935484,6.241815,7.685892,3.415573,...,273.876833,1.0,0.411601,31.0,280.0,45.0,310.0,NaN,NaN,COMPLETE
134,134,0.444182,0.012353,170,PyTorch/POT CUDA,75.470879,0.967742,6.251142,7.683039,3.413492,...,168.319648,1.0,0.403297,31.0,170.0,45.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
35,35,0.444945,0.012403,240,PyTorch/POT CUDA,77.649955,0.935484,6.247455,7.680139,3.420360,...,235.806452,1.0,0.410071,31.0,240.0,45.0,310.0,NaN,NaN,COMPLETE


,meilleur essai
experiment,kmeans_sinkhorn
trial,146
mean_intra_inter_ratio,0.441694
n_clusters,155
epsilon,0.012951
backend,PyTorch/POT CUDA
elapsed_s,76.024927
intra_identity_top1_accuracy,0.935484
mean_displacement_mm,6.24478
mean_inter_identity_distance_mm,7.703292


## Analyse des essais observés

Analyse des essais `COMPLETE` déjà présents dans `study` ; le score intra/inter-identité est à minimiser. Comme dans le notebook 02, les tableaux suivants distinguent l'optimum strict des compromis proches du meilleur score.


In [7]:
df = study.trials_dataframe()[[
    "number", "value", "params_epsilon", "params_n_clusters",
    "user_attrs_intra_identity_top1_accuracy",
]]
df = df.dropna(subset=["value"]).rename(columns={
    "number": "trial",
    "value": "score",
    "params_epsilon": "epsilon",
    "params_n_clusters": "n_clusters",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = df["score"].min()

display(
    df.sort_values("score")[["trial", "score", "epsilon", "n_clusters", "reid_accuracy"]]
    .style.format({
        "score": "{:.6f}", "epsilon": "{:.6g}",
        "reid_accuracy": "{:.0%}",
    })
)

reid = study.best_trial.user_attrs
if "reid_valid_bundles" in reid:
    print("RE-ID validée :", reid["reid_valid_bundles"])
    print("RE-ID échouée :", reid["reid_failed_bundles"])

rows = []
for threshold in (1, 2, 5):
    candidates = df[df["score"] <= best_score * (1 + threshold / 100)]
    # À performance comparable, epsilon élevé favorise un transport plus
    # régularisé et généralement plus stable numériquement.
    row = candidates.sort_values(
        ["epsilon", "score"], ascending=[False, True]
    ).iloc[0]
    rows.append([
        threshold, int(row.trial), row.score,
        100 * (row.score / best_score - 1),
        row.epsilon, int(row.n_clusters), row.reid_accuracy,
    ])
tradeoff = pd.DataFrame(rows, columns=[
    "seuil (%)", "trial", "score", "écart relatif (%)",
    "epsilon", "n_clusters", "reid_accuracy",
])
display(
    tradeoff.style.format({
        "score": "{:.6f}", "écart relatif (%)": "{:.2f}",
        "epsilon": "{:.6g}", "reid_accuracy": "{:.0%}",
    })
)


,trial,score,epsilon,n_clusters,reid_accuracy
146,146,0.441694,0.0129508,155,94%
132,132,0.442375,0.0134965,165,94%
118,118,0.443212,0.0120421,225,94%
148,148,0.443369,0.0130885,185,94%
139,139,0.443602,0.0126033,180,94%
74,74,0.443683,0.012519,145,94%
140,140,0.443711,0.0127859,180,94%
48,48,0.443953,0.0125122,280,94%
134,134,0.444182,0.0123533,170,97%
35,35,0.444945,0.0124026,240,94%


RE-ID validée : ['tractosearch_nn_8_0mm_all_AF_L_m', 'tractosearch_nn_8_0mm_all_AF_R_m', 'tractosearch_nn_8_0mm_all_CC_1_m', 'tractosearch_nn_8_0mm_all_CC_2a_m', 'tractosearch_nn_8_0mm_all_CC_2b_m', 'tractosearch_nn_8_0mm_all_CC_3_m', 'tractosearch_nn_8_0mm_all_CC_4_m', 'tractosearch_nn_8_0mm_all_CC_5_m', 'tractosearch_nn_8_0mm_all_CC_6_m', 'tractosearch_nn_8_0mm_all_CC_7_m', 'tractosearch_nn_8_0mm_all_CG_L_m', 'tractosearch_nn_8_0mm_all_CG_R_m', 'tractosearch_nn_8_0mm_all_CST_R_m', 'tractosearch_nn_8_0mm_all_ICP_L_m', 'tractosearch_nn_8_0mm_all_ICP_R_m', 'tractosearch_nn_8_0mm_all_IFOF_L_m', 'tractosearch_nn_8_0mm_all_ILF_L_m', 'tractosearch_nn_8_0mm_all_ILF_R_m', 'tractosearch_nn_8_0mm_all_MCP_m', 'tractosearch_nn_8_0mm_all_OR_L_m', 'tractosearch_nn_8_0mm_all_OR_R_m', 'tractosearch_nn_8_0mm_all_SLF_1_L_m', 'tractosearch_nn_8_0mm_all_SLF_1_R_m', 'tractosearch_nn_8_0mm_all_SLF_2_L_m', 'tractosearch_nn_8_0mm_all_SLF_2_R_m', 'tractosearch_nn_8_0mm_all_SLF_3_L_m', 'tractosearch_nn_8_0mm_a

,seuil (%),trial,score,écart relatif (%),epsilon,n_clusters,reid_accuracy
0,1,130,0.445571,0.88,0.017737,155,94%
1,2,117,0.447351,1.28,0.0185389,215,94%
2,5,127,0.453404,2.65,0.0261423,95,94%


## Visualisations Optuna

Le contour Optuna montre l'effet conjoint de `epsilon` et `n_clusters` sur le score.


In [8]:
from plotly.io import show

fig = optuna.visualization.plot_contour(
    study, params=["epsilon", "n_clusters"]
)
show(fig)


## Interprétation des compromis observés

L'étude finale contient **80 essais `COMPLETE`**. Elle compte aussi 61 essais `PRUNED` pour non-convergence de Sinkhorn, 6 essais `FAIL` et 2 anciens essais `RUNNING`; seuls les essais `COMPLETE` sont utilisés pour le classement.

Le **trial 146** est le nouvel optimum observé (`score=0,441694`, `n_clusters=155`, `epsilon≈0,012951`). Son exactitude RE-ID Top-1 est de **93,55 %** (29 bundles sur 31); les deux échecs restent `CST_L` et `IFOF_R`.

Parmi les solutions proches de l'optimum, une valeur d'`epsilon` plus élevée offre davantage de régularisation et généralement une meilleure stabilité numérique :

- à moins de 1 % : trial 130, `score=0,445571`, `epsilon≈0,017737`, `n_clusters=155` (+0,88 %);
- à moins de 2 % : trial 117, `score=0,447351`, `epsilon≈0,018539`, `n_clusters=215` (+1,28 %);
- à moins de 5 % : trial 127, `score=0,453404`, `epsilon≈0,026142`, `n_clusters=95` (+2,65 %).

- Selon la règle RE-ID commune, privilégier le **trial 134** (`n_clusters=170`, `epsilon≈0,012353`) : Top-1 96,77 %, rang moyen 1,06 et ratio 0,444182.
- Pour un **compromis très proche de l'optimum avec davantage de régularisation**, considérer le trial 130.
- Pour les analyses anatomiques et une future initialisation **LDDMM**, le projet retient le **trial 130 par défaut** : transport plus lisse et stable pour seulement +0,88 % sur le score.
- `epsilon` ne représente pas une couverture anatomique comme `mass` dans Partial OT : le compromis doit donc être interprété comme régularisation versus score, et non comme quantité de faisceau conservée.
